# SQL 문제은행 — 2026년 **2회** 유형

> 2026년 2회 실기에서 새로 드러난 함정만 모았습니다.
> 이 회차의 SQL 문제는 **상관 서브쿼리**, **안티 조인**(OUTER JOIN + IS NULL), `CREATE DOMAIN`, 빈 집합의 `AVG` 를 물었습니다.
>
> 1회가 "언어가 **언제 무엇을 결정하는가**"였다면,
> 2회는 **"끝까지 손으로 따라갈 수 있는가"**를 물었습니다.
>
> **규칙**
> 1. 정답 토글을 열기 전에 **반드시 종이에 적는다.**
> 2. 재귀와 트리는 **호출 트리를 그려서** 아래에서 위로 값을 채운다. 암산하면 반드시 틀린다.
> 3. 먼저 `2026_1회_유형` 폴더를 끝낸 뒤에 이쪽으로 온다.

---

## Q1. 상관 서브쿼리 (26년2회 8번 유형)

**[A] 테이블**

| id | x |
|---|---|
| 1 | 10 |
| 2 | 20 |
| 3 | 30 |
| 4 | 40 |

**[B] 테이블**

| id | y |
|---|---|
| 1 | 5 |
| 1 | 15 |
| 2 | 20 |
| 3 | 35 |
| 5 | 50 |

```sql
SELECT COUNT(*)
FROM A
WHERE x > (
    SELECT AVG(y)
    FROM B
    WHERE B.id IN (
        SELECT A2.id
        FROM A A2
        WHERE A2.x < A.x
    )
);
```

**실행 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
COUNT(*)
--------
3       
```

**상관 서브쿼리**다 — 안쪽 쿼리가 바깥의 `A.x` 를 참조하므로 **A의 행마다 따로 계산**해야 한다.
한 번에 풀려 하지 말고 4행을 각각 표로 정리한다.

| A.x | 나보다 작은 A2.id | B에서 뽑히는 y | AVG(y) | x > AVG? |
|---|---|---|---|---|
| 10 | 없음 | 없음 | **NULL** | ❌ |
| 20 | 1 | 5, 15 | 10 | ✅ |
| 30 | 1, 2 | 5, 15, 20 | 13.33 | ✅ |
| 40 | 1, 2, 3 | 5, 15, 20, 35 | 18.75 | ✅ |

→ **3**

첫 행이 핵심이다. 서브쿼리가 **빈 집합**이면 `AVG` 는 0이 아니라 **NULL** 이고,
`10 > NULL` 은 참이 아니라 **알 수 없음(UNKNOWN)** 이라 WHERE를 통과하지 못한다.

</details>

## Q2. OUTER JOIN + IS NULL — 안티 조인 (26년2회 15번 유형)

**[A] 테이블**

| id | v |
|---|---|
| 1 | 10 |
| 1 | 20 |
| 2 | 30 |

**[B] 테이블**

| id | w |
|---|---|
| 1 | 100 |
| 3 | 300 |
| 4 | 400 |

```sql
SELECT COUNT(*) AS Result
FROM A
RIGHT OUTER JOIN B ON A.id = B.id
WHERE A.id IS NULL;
```

**실행 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
Result
------
1     
```

`RIGHT OUTER JOIN B` 이므로 **B의 모든 행이 남는다.** 조인 결과를 먼저 그린다.

| B.id | 매칭되는 A 행 | 결과 행 수 | A.id |
|---|---|---|---|
| 1 | (1,10), (1,20) 두 개 | 2 | 1 |
| 3 | 없음 | 1 | **NULL** |
| 4 | 없음 | 1 | **NULL** |

`WHERE A.id IS NULL` 은 **매칭에 실패한 행만** 남긴다 → id 3, 4 → **2**

이 패턴(OUTER JOIN + IS NULL)을 **안티 조인**이라 하며,
"B에는 있는데 A에는 없는 것"을 찾는 표준 관용구다. `NOT IN` / `NOT EXISTS` 와 같은 목적이다.

> B.id = 1이 **2행으로 늘어나는 것**도 함정이다. 조인은 매칭되는 만큼 행이 곱해진다.

</details>

## Q3. 도메인 무결성 — CREATE DOMAIN (26년2회 14번 유형)

다음 SQL은 SEASON 도메인의 값이 지정된 계절 중 하나만 되도록 **도메인 무결성**을 보장한다.
빈칸 ①에 들어갈 SQL 키워드를 쓰시오.

```sql
CREATE DOMAIN SEASON AS VARCHAR(6)
    ( ① ) (VALUE IN ('spring', 'summer', 'autumn', 'winter'));
```

또한 무결성 제약의 3가지 종류를 쓰고 각각을 한 줄로 설명하시오.

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
① CHECK

무결성 3종
- 개체 무결성 : 기본키는 NULL이 될 수 없고 중복될 수 없다
- 참조 무결성 : 외래키는 참조하는 릴레이션의 기본키 값이거나 NULL이어야 한다
- 도메인 무결성 : 각 속성의 값은 정의된 도메인(자료형·범위·형식)에 속해야 한다
```

`CREATE DOMAIN` 은 **재사용 가능한 사용자 정의 자료형**을 만드는 구문이고,
값의 범위를 제한하는 키워드는 테이블 제약과 똑같이 **CHECK** 다.

`VALUE` 는 그 도메인에 들어올 값을 가리키는 예약어다 (컬럼명이 아니다).

```sql
CREATE DOMAIN SEASON AS VARCHAR(6)
    CHECK (VALUE IN ('spring','summer','autumn','winter'));
```

> 도메인 제약은 **도메인 무결성**에 해당한다. 셋 중 무엇인지 함께 묻는 경우가 많다.

</details>

## Q4. LIKE 와 ORDER BY 빈칸 (26년2회 19번 유형)

**[회사] 테이블**

| 이름 | 부서 | 연봉 |
|---|---|---|
| 이서연 | 영업부 | 6200 |
| 박도윤 | 영업부 | 4800 |
| 이하준 | 기획부 | 3400 |
| 최지우 | 총무부 | 3100 |
| 이유나 | 기획부 | 5300 |

이름이 '이'로 시작하는 사원을 **이름의 내림차순**으로 조회하려 한다. 빈칸을 채우시오.

```sql
SELECT *
FROM 회사
WHERE 이름 LIKE '( ① )'
ORDER BY 이름 ( ② );
```

그리고 **실행 결과**도 쓰시오.

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
① 이%
② DESC

실행 결과

이름  | 부서  | 연봉  
----+-----+-----
이하준 | 기획부 | 3400
이유나 | 기획부 | 5300
이서연 | 영업부 | 6200
```

- ① **`이%`** — `%` 는 **0글자 이상** 아무 문자열. '이'로 시작하기만 하면 된다.
- ② **`DESC`** — 내림차순. 생략하면 기본값이 `ASC`(오름차순)다.

와일드카드를 구분할 것.

| 패턴 | 뜻 |
|---|---|
| `'이%'` | 이로 **시작** |
| `'%이'` | 이로 **끝남** |
| `'%이%'` | 이를 **포함** |
| `'이_'` | 이 + **정확히 한 글자** (두 글자 이름만) |

한글 내림차순은 가나다 역순이라 이하준 → 이유나 → 이서연 순이 된다.

</details>

## Q5. 집계함수와 NULL — 빈 집합의 AVG

**[B] 테이블** (위 문제와 동일)

| id | y |
|---|---|
| 1 | 5 |
| 1 | 15 |
| 2 | 20 |
| 3 | 35 |
| 5 | 50 |

```sql
SELECT COUNT(*), COUNT(y), AVG(y),
       (SELECT AVG(y) FROM B WHERE id = 99) AS 빈집합
FROM B;
```

**실행 결과를 쓰시오.**

<details>
<summary><b>👉 정답 · 해설 보기</b></summary>

```
COUNT(*) | COUNT(y) | AVG(y) | 빈집합 
---------+----------+--------+-----
5        | 5        | 25.0   | NULL
```

집계함수가 **조회할 행이 하나도 없을 때** 무엇을 반환하는지가 핵심이다.

| 함수 | 빈 집합일 때 |
|---|---|
| `COUNT(*)` / `COUNT(컬럼)` | **0** |
| `SUM` / `AVG` / `MAX` / `MIN` | **NULL** |

`COUNT` 만 0이고 나머지는 전부 **NULL** 이다. 0이 아니다.

그래서 `x > (SELECT AVG(...) ...)` 같은 비교에서 서브쿼리가 비면
`x > NULL` 이 되어 **참이 될 수 없다** — 26년 2회 8번의 첫 행이 정확히 이 경우였다.

</details>